In [8]:
import os, sys
from pathlib import Path

def find_project_root(start=None, markers=("openpoints", "cfgs")):
    p = Path(start or os.getcwd()).resolve()
    for parent in [p] + list(p.parents):
        if all((parent / m).exists() for m in markers):
            return parent
    raise RuntimeError(f"Cannot find project root from {p}, markers={markers}")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("cwd =", os.getcwd())
print("ROOT =", ROOT)
print("sys.path[0] =", sys.path[0])


cwd = /workspace/PointNeXt/quant
ROOT = /workspace/PointNeXt
sys.path[0] = /workspace/PointNeXt


In [9]:
"""
PTQ (Post-Training Quantization) for PointNeXt-S on S3DIS.

Uses prepare_qat() to insert FakeQuantize modules, but does NOT train.
Phase 1: disable fake_quant -> calibrate (pure observer, like standard PTQ)
Phase 2: enable  fake_quant -> evaluate  (true INT8-sim accuracy)

Usage:
    conda activate pointnext
    python quant/quant_wrapper.py
"""
    
import os, time, logging
import numpy as np
import torch
import torch.quantization as quant
from openpoints.models import build_model_from_cfg
from openpoints.dataset import build_dataloader_from_cfg, get_features_by_keys
from openpoints.utils import EasyConfig, ConfusionMatrix, get_mious
from openpoints.models.layers.quant_utils import (
    swap_custom_convs_to_standard,
    fuse_convbn_modules,
    disable_quantization_for_geometry,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ======================= config =======================
CFG_PATH = str (ROOT / "cfgs/s3dis/pointnext-s_kdtree_simple_128.yaml")
CKPT_PATH = str( ROOT /
    "log/s3dis/"
    "s3dis-train-pointnext-s_kdtree_simple_128"
    "-ngpus4-20260110-192922-Vfdejy4CdHAKfmY35stCtJ"
    "/checkpoint/"
    "s3dis-train-pointnext-s_kdtree_simple_128"
    "-ngpus4-20260110-192922-Vfdejy4CdHAKfmY35stCtJ"
    "_ckpt_best.pth"
)
BACKEND = "fbgemm"
NUM_CALIB_BATCHES = 30
OUTPUT_DIR = "quant/output"


def evaluate(model, val_loader, cfg, device="cuda", tag="FP32"):
    """Evaluate segmentation mIoU / OA / mAcc on full val set."""
    model.eval()
    model.to(device)
    num_classes = cfg.num_classes
    ignore_index = cfg.get("ignore_index", None)
    feature_keys = cfg.get("feature_keys", "pos")
    cm = ConfusionMatrix(num_classes=num_classes, ignore_index=ignore_index)
    t0 = time.time()
    with torch.no_grad():
        for i, data in enumerate(val_loader):
            for k in data:
                data[k] = data[k].to(device, non_blocking=True)
            data["x"] = get_features_by_keys(data, feature_keys)
            logits = model(data)
            target = data["y"].squeeze(-1)
            cm.update(logits.argmax(dim=1), target)
    miou, macc, oa, ious, accs = get_mious(cm.tp, cm.union, cm.count)
    elapsed = time.time() - t0
    logger.info("[%s] OA=%.2f  mAcc=%.2f  mIoU=%.2f  (%.1fs)", tag, oa, macc, miou, elapsed)
    with np.printoptions(precision=2, suppress=True):
        logger.info("[%s] IoU per class: %s", tag, ious)
    return miou, macc, oa, ious, accs


In [10]:
# ===== 1. build model + load weights =====
logger.info("Config: %s", CFG_PATH)
cfg = EasyConfig()
cfg.load(CFG_PATH, recursive=True)
model = build_model_from_cfg(cfg.model)
logger.info("Checkpoint: %s", CKPT_PATH)
ckpt = torch.load(CKPT_PATH, map_location="cpu")
state = ckpt["model"] if "model" in ckpt else ckpt
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:
    logger.warning("Missing keys (%d): %s ...", len(missing), missing[:3])
if unexpected:
    logger.warning("Unexpected keys (%d): %s ...", len(unexpected), unexpected[:3])
logger.info("Loaded (epoch=%s, best_val=%s)", ckpt.get("epoch", "?"), ckpt.get("best_val", "?"))
model.eval()
# ===== 2. build val dataloader =====
logger.info("Building S3DIS val dataloader ...")
val_loader = build_dataloader_from_cfg(
    cfg.get("val_batch_size", cfg.get("batch_size", 1)),
    cfg.dataset, cfg.dataloader,
    datatransforms_cfg=cfg.datatransforms,
    split="val", distributed=False,
)
logger.info("Val set: %d samples", len(val_loader.dataset))
# ===== 3. FP32 baseline evaluation =====
logger.info("=" * 60)
logger.info("Evaluating FP32 baseline on GPU ...")
model.cuda()
fp32_miou, fp32_macc, fp32_oa, fp32_ious, _ = evaluate(
    model, val_loader, cfg, device="cuda", tag="FP32"
)
# ===== 4. quantization preparation =====
model.cpu().eval()
logger.info("=" * 60)
logger.info("[Step 0] swap custom Conv to standard")
swap_custom_convs_to_standard(model)
logger.info("[Step 1] fuse Conv-BN-ReLU")
fuse_convbn_modules(model)
# ---- 4a. evaluate fused model (shows BN-folding error only) ----
logger.info("=" * 60)
logger.info("Evaluating FP32-fused model (after BN-folding, before quant) ...")
model.cuda()
fused_miou, fused_macc, fused_oa, _, _ = evaluate(
    model, val_loader, cfg, device="cuda", tag="FP32-fused"
)
logger.info("BN-folding delta:  mIoU %+.2f", fused_miou - fp32_miou)
model.cpu().eval()
logger.info("[Step 2] disable quant for geometry ops")
disable_quantization_for_geometry(model)
logger.info("[Step 3] set QAT qconfig (backend=%s)", BACKEND)
torch.backends.quantized.engine = BACKEND
model.qconfig = quant.get_default_qat_qconfig(BACKEND)
logger.info("[Step 4] prepare_qat (insert FakeQuantize modules)")
model.train()
quant.prepare_qat(model, inplace=True)
# ===== 5. calibration (Phase 1: disable fake_quant, observers only) =====
# Disable fake_quant so observers see raw FP32 activations
# (standard PTQ calibration behavior).
logger.info("[Phase 1] Disable FakeQuantize, enable Observer -> calibrate")
model.apply(quant.disable_fake_quant)
model.apply(quant.enable_observer)
model.eval()
model.cuda()
feature_keys = cfg.get("feature_keys", "pos")
logger.info("Calibrating %d batches on GPU ...", NUM_CALIB_BATCHES)
t0 = time.time()
with torch.no_grad():
    for i, data in enumerate(val_loader):
        if i >= NUM_CALIB_BATCHES:
            break
        for k in data:
            data[k] = data[k].cuda(non_blocking=True)
        data["x"] = get_features_by_keys(data, feature_keys)
        model(data)
        if (i + 1) % 10 == 0:
            logger.info("  batch %d/%d", i + 1, NUM_CALIB_BATCHES)
elapsed = time.time() - t0
logger.info("Calibration done in %.1fs (%d batches)", elapsed,
            min(NUM_CALIB_BATCHES, len(val_loader)))
# ===== 6. evaluate with FakeQuantize enabled (true INT8-sim) =====
# Phase 2: Enable fake_quant (simulates INT8 clamp+round-trip),
#          disable observer updates (freeze scale/zero_point).
# Model stays on GPU because geometry ops are CUDA-only.
logger.info("=" * 60)
logger.info("[Phase 2] Enable FakeQuantize, disable Observer -> evaluate")
model.apply(quant.enable_fake_quant)
model.apply(quant.disable_observer)
int8_miou, int8_macc, int8_oa, int8_ious, _ = evaluate(
    model, val_loader, cfg, device="cuda", tag="INT8-sim"
)
# ===== 7. convert to actual INT8 (for saving) =====
logger.info("Converting to real INT8 (for export only) ...")
model.cpu().eval()
quant.convert(model, inplace=True)
q_count = sum(1 for _, m in model.named_modules()
              if "quantized" in type(m).__module__)
logger.info("Quantized modules: %d", q_count)
# ===== 8. summary =====
logger.info("=" * 60)
logger.info("SUMMARY")
logger.info("-" * 60)
logger.info("mIoU:  FP32=%.2f  FP32-fused=%.2f  INT8-sim=%.2f",
            fp32_miou, fused_miou, int8_miou)
logger.info("  BN-folding delta:   %+.2f", fused_miou - fp32_miou)
logger.info("  Quantization delta: %+.2f  (INT8-sim vs FP32-fused)",
            int8_miou - fused_miou)
logger.info("  Total delta:        %+.2f  (INT8-sim vs FP32)",
            int8_miou - fp32_miou)
logger.info("-" * 60)
logger.info("OA:    FP32=%.2f  ->  INT8-sim=%.2f  (delta=%+.2f)",
            fp32_oa, int8_oa, int8_oa - fp32_oa)
logger.info("mAcc:  FP32=%.2f  ->  INT8-sim=%.2f  (delta=%+.2f)",
            fp32_macc, int8_macc, int8_macc - fp32_macc)
# ===== 9. save =====
os.makedirs(OUTPUT_DIR, exist_ok=True)
save_path = os.path.join(OUTPUT_DIR, "pointnext_s_simple128_ptq_int8.pth")
torch.save({
    "model": model.state_dict(),
    "config": CFG_PATH,
    "backend": BACKEND,
    "calib_batches": NUM_CALIB_BATCHES,
    "quantized_modules": q_count,
    "fp32_miou": fp32_miou, "fp32_oa": fp32_oa, "fp32_macc": fp32_macc,
    "fused_miou": fused_miou, "fused_oa": fused_oa, "fused_macc": fused_macc,
    "int8_miou": int8_miou, "int8_oa": int8_oa, "int8_macc": int8_macc,
}, save_path)
logger.info("Saved: %s", save_path)
logger.info("All done!")

2026-03-10 13:33:28,769 INFO Config: /workspace/PointNeXt/cfgs/s3dis/pointnext-s_kdtree_simple_128.yaml
2026-03-10 13:33:28,793 INFO radius: [[0.1], [0.1], [0.2], [0.4], [0.8]],
 nsample: [[32], [32], [32], [32], [32]]
2026-03-10 13:33:28,796 INFO NAME: ballquery
normalize_dp: True
radius: 0.1
nsample: 32
2026-03-10 13:33:28,799 INFO NAME: ballquery
normalize_dp: True
radius: 0.2
nsample: 32
2026-03-10 13:33:28,802 INFO NAME: ballquery
normalize_dp: True
radius: 0.4
nsample: 32
2026-03-10 13:33:28,806 INFO NAME: ballquery
normalize_dp: True
radius: 0.8
nsample: 32
2026-03-10 13:33:28,813 INFO Checkpoint: /workspace/PointNeXt/log/s3dis/s3dis-train-pointnext-s_kdtree_simple_128-ngpus4-20260110-192922-Vfdejy4CdHAKfmY35stCtJ/checkpoint/s3dis-train-pointnext-s_kdtree_simple_128-ngpus4-20260110-192922-Vfdejy4CdHAKfmY35stCtJ_ckpt_best.pth


Create grouper: ballquery, radius: 0.1, nsample: 32, args: normalize_dp: True
Create grouper: ballquery, radius: 0.2, nsample: 32, args: normalize_dp: True
Create grouper: ballquery, radius: 0.4, nsample: 32, args: normalize_dp: True
Create grouper: ballquery, radius: 0.8, nsample: 32, args: normalize_dp: True


2026-03-10 13:33:29,171 INFO Loaded (epoch=67, best_val=57.37431335449219)
2026-03-10 13:33:29,173 INFO Building S3DIS val dataloader ...
2026-03-10 13:33:29,285 INFO 
Totally 68 samples in val set
2026-03-10 13:33:29,287 INFO Val set: 68 samples
2026-03-10 13:33:29,287 INFO ============================================================
2026-03-10 13:33:29,288 INFO Evaluating FP32 baseline on GPU ...


/workspace/dataset/s3disfull/processed/s3dis_val_area5_0.040_None.pkl load successfully


2026-03-10 13:33:35,181 INFO [Sampler] kdtree_simple_sample | B=1, N=47355, npoint=11838, leaf_size=128, strategy=random, proportional=True | recursive median-split + leaf random sampling
2026-03-10 13:33:53,636 INFO [FP32] OA=84.48  mAcc=64.07  mIoU=56.54  (20.8s)
2026-03-10 13:33:53,639 INFO [FP32] IoU per class: [90.52 96.58 74.9   0.   20.34 44.48 35.11 74.65 80.53 47.43 68.53 53.87
 48.08]
2026-03-10 13:33:53,649 INFO ============================================================
2026-03-10 13:33:53,650 INFO [Step 0] swap custom Conv to standard
2026-03-10 13:33:53,651 INFO [Step 1] fuse Conv-BN-ReLU
2026-03-10 13:33:53,714 INFO ============================================================
2026-03-10 13:33:53,716 INFO Evaluating FP32-fused model (after BN-folding, before quant) ...
2026-03-10 13:34:12,277 INFO [FP32-fused] OA=84.56  mAcc=64.72  mIoU=57.19  (18.6s)
2026-03-10 13:34:12,284 INFO [FP32-fused] IoU per class: [90.37 96.63 75.02  0.09 19.9  44.1  37.1  75.78 79.88 54.45 68.

In [11]:
for name, m in model.named_modules():
    if len(list(m.children())) == 0:  # leaf module
        print(f"{name:70s}  {type(m).__name__}")

encoder.encoder.0.0.convs.0.0                                           Conv1d
encoder.encoder.1.0.skipconv.0                                          Conv1d
encoder.encoder.1.0.act                                                 ReLU
encoder.encoder.1.0.quant_skip                                          Quantize
encoder.encoder.1.0.convs.0.0                                           ConvReLU2d
encoder.encoder.1.0.convs.0.1                                           Identity
encoder.encoder.1.0.convs.0.2                                           Identity
encoder.encoder.1.0.convs.1.0                                           Conv2d
encoder.encoder.1.0.convs.1.1                                           Identity
encoder.encoder.1.0.grouper                                             QueryAndGroup
encoder.encoder.1.0.pool                                                MaxPool
encoder.encoder.1.0.qadd.ff.activation_post_process                     Identity
encoder.encoder.1.0.dequant_feat

In [12]:
print(model)

BaseSeg(
  (encoder): PointNextEncoder(
    (encoder): Sequential(
      (0): Sequential(
        (0): SetAbstraction(
          (convs): Sequential(
            (0): Sequential(
              (0): QuantizedConv1d(4, 32, kernel_size=(1,), stride=(1,), scale=0.02864639274775982, zero_point=58)
            )
          )
        )
      )
      (1): Sequential(
        (0): SetAbstraction(
          (skipconv): Sequential(
            (0): QuantizedConv1d(32, 64, kernel_size=(1,), stride=(1,), scale=0.041021332144737244, zero_point=89)
          )
          (act): ReLU(inplace=True)
          (quant_skip): Quantize(scale=tensor([0.0278]), zero_point=tensor([56]), dtype=torch.quint8)
          (convs): Sequential(
            (0): Sequential(
              (0): QuantizedConvReLU2d(35, 32, kernel_size=(1, 1), stride=(1, 1), scale=0.04379672184586525, zero_point=0)
              (1): Identity()
              (2): Identity()
            )
            (1): Sequential(
              (0): Quanti

In [18]:
qconv = None
qname = None
for name, m in model.named_modules():
    if type(m).__module__.startswith("torch.nn.quantized") and type(m).__name__ == "Conv1d":
        qconv, qname = m, name
        break

print("picked:", qname, type(qconv))

wq = qconv.weight()
print("shape:", tuple(wq.shape), "dtype:", wq.dtype, "is_quant:", wq.is_quantized)
print("int min/max:", int(wq.int_repr().min()), int(wq.int_repr().max()))

picked: encoder.encoder.0.0.convs.0.0 <class 'torch.nn.quantized.modules.conv.Conv1d'>
shape: (32, 4, 1) dtype: torch.qint8 is_quant: True
int min/max: -128 127


In [21]:
import torch.quantization as quant

qconfig = quant.get_default_qat_qconfig('qnnpack')

print("=== QConfig ===")
print(f"Activation: {qconfig.activation}")
print(f"Weight:     {qconfig.weight}")

# 实例化 observer 看具体参数
act_obs = qconfig.activation()
wt_obs  = qconfig.weight()

print("\n=== Activation Observer ===")
print(act_obs)
print(f"  dtype:      {act_obs.dtype}")
print(f"  quant_min:  {act_obs.quant_min}")
print(f"  quant_max:  {act_obs.quant_max}")
print(f"  qscheme:    {act_obs.qscheme}")

print("\n=== Weight Observer ===")
print(wt_obs)
print(f"  dtype:      {wt_obs.dtype}")
print(f"  quant_min:  {wt_obs.quant_min}")
print(f"  quant_max:  {wt_obs.quant_max}")
print(f"  qscheme:    {wt_obs.qscheme}")

=== QConfig ===
Activation: functools.partial(<class 'torch.ao.quantization.fake_quantize.FusedMovingAvgObsFakeQuantize'>, observer=<class 'torch.ao.quantization.observer.MovingAverageMinMaxObserver'>, quant_min=0, quant_max=255, reduce_range=False){}
Weight:     functools.partial(<class 'torch.ao.quantization.fake_quantize.FusedMovingAvgObsFakeQuantize'>, observer=<class 'torch.ao.quantization.observer.MovingAverageMinMaxObserver'>, quant_min=-128, quant_max=127, dtype=torch.qint8, qscheme=torch.per_tensor_symmetric){}

=== Activation Observer ===
FusedMovingAvgObsFakeQuantize(
  fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, reduce_range=False
  (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
)
  dtype:      torch.quint8
  quant_min:  0
  quant_max:  255
  qscheme:    torch.per_tensor_affine

=